# 非参数 Bootstrap（对数正态）

> 对应代码：`MathStatsCode/code_in_notes/Chap.15/bootstrap_lognormal_np.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用 bootstrap 命令对对数正态样本做非参数再抽样，并用 estat bootstrap 查看结果。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear all`、`set more off`、`set seed 1000`。与参数化版本使用相同的数据生成过程：n=10、对数 N(2,2)、x=exp(z)。


In [ ]:
%%stata
clear all
set more off
set seed 1000
set obs 10


**产生样本**

定义用于 Bootstrap 的**估计程序** `lognormalest`：`syntax varlist` 接受变量后求均值、用 `log(r(mean))-1` 得到 μ 的估计，并通过 `return scalar mu=` 返回。`bootstrap` 命令要求把估计过程封装成这样的程序。


In [ ]:
%%stata
local mu=2
gen z=rnormal(`mu',sqrt(2))
gen x=exp(z)


**定义估计的program**

执行**非参数 Bootstrap**：`bootstrap mu=r(mu), reps(1000): lognormalest x`——与参数化版本的区别在于：非参数 Bootstrap **不对分布做任何假设**，而是从原始样本中“有放回”地重抽 n 个观测，每次重算估计量；`reps(1000)` 指定重复次数。


In [ ]:
%%stata
cap drop program lognormalest
program define lognormalest, rclass
	syntax varname
	quietly {
		su `varlist'
		local mu=log(r(mean))-1
		return scalar mu=`mu'
	}
end


**bootstrap**

`estat bootstrap`：输出 Bootstrap 的汇总结果（观测值、偏差、标准误以及正态/百分位/t 三种置信区间），并可与理论标准误对比。这体现了 Stata 内置 `bootstrap` 命令的便利：它自动处理重抽样、结果汇总与置信区间计算。


In [ ]:
%%stata
local B=1000
bootstrap mu=r(mu), reps(`B'): lognormalest x
estat bootstrap 
